[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/github-actions-certified/notebooks/day-05-secrets-and-security.ipynb#scrollTo=11223344)

---
# Day 5 · Secrets Management and Security Best Practices
**certified-journeys / github-actions-certified** · Day 5 · Security Hardening

> **Goal for today:** Know how to store secrets safely, consume them in workflows without leaking them, scope `GITHUB_TOKEN` permissions to least privilege, and understand how OIDC eliminates long-lived credentials entirely.


In [ ]:
%pip install -q pyyaml


## Step 1 · How GitHub Secrets Work

Repository secrets are encrypted at rest with Libsodium and only decrypted inside the runner process during a workflow run. They are **never printed** in logs — Actions automatically replaces any occurrence of the secret value with `***`.

| Scope | Set in | Available to |
|---|---|---|
| Repository | Settings → Secrets → Actions | That repo only |
| Environment | Settings → Environments | Jobs that specify `environment:` |
| Organization | Org Settings → Secrets | Repos granted access |

Secrets are referenced as `${{ secrets.SECRET_NAME }}` in workflow YAML.


In [ ]:
import yaml

# Workflow that consumes a secret — notice the secret goes into env, NOT args
secrets_workflow = {
    "name": "Deploy with secret",
    "on": {"push": {"branches": ["main"]}},
    "jobs": {
        "deploy": {
            "runs-on": "ubuntu-latest",
            "steps": [
                {"uses": "actions/checkout@v4"},
                {
                    "name": "Publish package",
                    # CORRECT: secret passed via environment variable
                    "env": {"PYPI_TOKEN": "${{ secrets.PYPI_TOKEN }}"},
                    # Never: run: twine upload --password ${{ secrets.PYPI_TOKEN }}
                    # (positional args are visible in process list)
                    "run": "twine upload dist/* --username __token__ --password $PYPI_TOKEN",
                },
            ],
        }
    },
}

print(yaml.dump(secrets_workflow, sort_keys=False))


**What just happened?**

- The secret is injected into the step's environment as `PYPI_TOKEN`, then referenced as a shell variable `$PYPI_TOKEN`.
- **Never** pass secrets as positional arguments — they appear in `/proc/<pid>/cmdline` and process listings.
- GitHub's log masker detects the secret value and replaces it with `***` anywhere it appears in stdout/stderr.


## Step 2 · Secret Masking and Anti-Patterns

Understanding *why* secrets can leak helps you avoid the traps:

| Anti-pattern | Why it's dangerous |
|---|---|
| `echo $SECRET` | Directly prints to stdout — masked, but still bad habit |
| Passing as CLI arg | Visible in process list and `ps aux` output |
| Storing in artifact | Artifacts are downloadable by anyone with repo access |
| Using in `run:` expression | `${{ secrets.X }}` in run blocks is evaluated before masking |
| Logging decoded base64 | Actions masks the raw value but not its decoded form |

The safe pattern: **env var → shell variable → subprocess stdin or env**.


In [ ]:
# Demonstrate the safe pattern vs dangerous patterns as YAML snippets

safe_step = {
    "name": "Safe — secret via env var",
    "env": {"DB_PASSWORD": "${{ secrets.DB_PASSWORD }}"},
    "run": "python scripts/migrate.py  # reads os.environ['DB_PASSWORD']",
}

dangerous_step = {
    "name": "DANGEROUS — secret interpolated in run string",
    # This expands the secret value into the shell command before masking can act
    "run": "python scripts/migrate.py --password ${{ secrets.DB_PASSWORD }}",
}

print("=== SAFE pattern ===")
print(yaml.dump(safe_step, sort_keys=False))

print("=== DANGEROUS pattern (do NOT use) ===")
print(yaml.dump(dangerous_step, sort_keys=False))

print("""
Rule: Never use ${{ secrets.X }} inside a 'run:' command string.
Always map secrets to 'env:' and read them as environment variables in your script.
""")


**What just happened?**

- The `env:` key injects the secret into the runner process environment **before** the shell command runs.
- In the dangerous pattern, `${{ secrets.DB_PASSWORD }}` is a template expression evaluated by Actions — the raw value gets embedded in the shell command string, visible in debug logs.
- **The golden rule:** `${{ secrets.X }}` belongs only in `env:`, `with:` (action inputs), or `if:` conditions — never in `run:` text.


## Step 3 · Scoping `GITHUB_TOKEN` to Least Privilege

`GITHUB_TOKEN` is auto-generated for every workflow run. Its default permissions are set at the org/repo level, but you can **narrow them per workflow or per job** using the `permissions:` key.

| Permission | Values | Use case |
|---|---|---|
| `contents` | `read` / `write` / `none` | Cloning, creating releases |
| `pull-requests` | `read` / `write` / `none` | Posting PR comments |
| `packages` | `read` / `write` / `none` | Publishing to GitHub Packages |
| `id-token` | `write` / `none` | Required for OIDC token exchange |

Setting `permissions:` at the **job level** is always safer than at the workflow level — it restricts the token only for that job's lifetime.


In [ ]:
# Least-privilege workflow: one job reads (CI), another writes (release)
least_priv_workflow = {
    "name": "Least-privilege CI + Release",
    "on": {"push": {"branches": ["main"]}},
    # Top-level default: deny everything
    "permissions": {},  # empty = no permissions granted by default
    "jobs": {
        "test": {
            "runs-on": "ubuntu-latest",
            # This job only needs to read the repo
            "permissions": {"contents": "read"},
            "steps": [
                {"uses": "actions/checkout@v4"},
                {"run": "pytest tests/"},
            ],
        },
        "release": {
            "runs-on": "ubuntu-latest",
            "needs": "test",
            # This job needs to write releases (upload assets)
            "permissions": {"contents": "write"},
            "steps": [
                {"uses": "actions/checkout@v4"},
                {
                    "name": "Create GitHub Release",
                    "uses": "softprops/action-gh-release@v2",
                    "with": {"files": "dist/*.whl"},
                },
            ],
        },
    },
}

print(yaml.dump(least_priv_workflow, sort_keys=False))


**What just happened?**

- The top-level `permissions: {}` sets a blanket deny — every job starts with zero access.
- The `test` job is granted **only** `contents: read` — enough to checkout and run tests.
- The `release` job is granted **only** `contents: write` — the minimum needed to upload release assets.
- If the `test` job were compromised, it couldn't create releases, write to the repo, or post comments.


## Step 4 · Security Hardening — Pinning Actions to Commit SHAs

Using `actions/checkout@v4` means your workflow trusts whatever code is at that tag. Tags are **mutable** — a malicious maintainer can move the tag. Pinning to a commit SHA makes the dependency immutable.

| Reference | Safety | Maintenance |
|---|---|---|
| `@v4` (tag) | Mutable — tag can be repointed | Easy to update |
| `@v4.1.1` (semver tag) | Still mutable | Slightly better |
| `@a6338ad4...` (SHA) | Immutable — content-addressed | Requires tooling to update |

For MLOps workflows that gate on model quality, supply chain integrity matters: a compromised action could silently lower your accuracy threshold or exfiltrate model weights.


In [ ]:
# Build a security-hardened workflow with SHA-pinned actions
# SHA values below are real pinned versions (verify before use in production)

# Map: action → pinned SHA (with human-readable tag in comment)
PINNED_ACTIONS = {
    "checkout": "actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683",  # v4.2.2
    "setup_python": "actions/setup-python@0b93645e9fea7318ecaed2b359559ac225c90a2b",  # v5.3.0
    "upload_artifact": "actions/upload-artifact@4cec3d8aa04e39d1a68397de0c4cd6fb9dce8ec1",  # v4.6.1
    "cache": "actions/cache@6849a6489940f00c2f30c0fb92c6274307ccb58a",  # v4.1.2
}

hardened_workflow = {
    "name": "SHA-pinned hardened workflow",
    "on": {"push": {"branches": ["main"]}},
    "permissions": {"contents": "read"},
    "jobs": {
        "ci": {
            "runs-on": "ubuntu-latest",
            "steps": [
                # Each action pinned to an immutable commit SHA
                {"uses": PINNED_ACTIONS["checkout"]},
                {
                    "uses": PINNED_ACTIONS["setup_python"],
                    "with": {"python-version": "3.12"},
                },
                {
                    "uses": PINNED_ACTIONS["cache"],
                    "with": {
                        "path": "~/.cache/pip",
                        "key": "${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}",
                    },
                },
                {"run": "pip install -r requirements.txt && pytest tests/"},
            ],
        }
    },
}

print(yaml.dump(hardened_workflow, sort_keys=False))


**What just happened?**

- Every `uses:` references a full commit SHA — the content is **content-addressed** and cannot change.
- Keep the tag as a comment (`# v4.2.2`) so humans know what version it is.
- Tools like **Dependabot** and **Renovate** can automate SHA updates so you don't fall behind on security patches.
- For internal/private actions where you control the repo, tags are acceptable — the risk is from third-party actions.


## Step 5 · OIDC — Keyless Cloud Authentication

OpenID Connect (OIDC) lets your workflow **request short-lived credentials from a cloud provider** without storing any long-lived secret. The flow:

1. GitHub generates a JWT signed by GitHub's OIDC provider for your repo/branch/workflow.
2. Your workflow requests this token (`id-token: write` permission required).
3. The cloud provider (AWS, GCP, Azure) verifies the JWT and issues short-lived credentials.
4. No secret is stored in the repository — the credential lives only for the duration of the job.

| Approach | Secret lifetime | Storage | Rotation |
|---|---|---|---|
| Long-lived key (old way) | Forever until rotated | GitHub Secrets | Manual |
| OIDC (modern way) | Minutes (job duration) | Nowhere | Automatic |


In [ ]:
# OIDC workflow example — AWS S3 upload without any stored access key
oidc_workflow = {
    "name": "OIDC — Deploy to S3 without stored credentials",
    "on": {"push": {"branches": ["main"]}},
    "jobs": {
        "deploy": {
            "runs-on": "ubuntu-latest",
            "permissions": {
                "contents": "read",
                # Required to request the OIDC token from GitHub
                "id-token": "write",
            },
            "steps": [
                {"uses": "actions/checkout@v4"},
                {
                    "name": "Configure AWS credentials via OIDC",
                    "uses": "aws-actions/configure-aws-credentials@v4",
                    "with": {
                        # IAM role that trusts GitHub OIDC — no access key stored anywhere
                        "role-to-assume": "arn:aws:iam::123456789012:role/GitHubActionsRole",
                        "aws-region": "us-east-1",
                    },
                },
                {
                    "name": "Upload model to S3",
                    "run": "aws s3 cp model.pkl s3://my-model-bucket/model.pkl",
                },
            ],
        }
    },
}

print(yaml.dump(oidc_workflow, sort_keys=False))

print("""
Why OIDC is better:
  ✓ No AWS_ACCESS_KEY_ID stored in GitHub Secrets
  ✓ Credentials expire when the job ends (minutes)
  ✓ IAM role conditions can restrict to specific branches/repos
  ✓ Audit trail in AWS CloudTrail includes the workflow run ID
""")


**What just happened?**

- `id-token: write` is the only permission needed — the action handles the token exchange.
- `aws-actions/configure-aws-credentials` calls `sts:AssumeRoleWithWebIdentity` with the JWT.
- The resulting credentials are exported as environment variables (`AWS_ACCESS_KEY_ID`, etc.) and live only for this job.
- **No secret rotation needed** — there's nothing to rotate because nothing is stored.


## Step 6 · Comprehensive Security Checklist Workflow

Let's build a function that generates a security-auditable workflow combining all the techniques from today: scoped permissions, SHA-pinned actions, secrets via env, and OIDC.


In [ ]:
def build_secure_ml_workflow(
    python_version: str = "3.12",
    use_oidc: bool = True,
    aws_role_arn: str = "arn:aws:iam::123456789012:role/GitHubActionsRole",
    aws_region: str = "us-east-1",
) -> dict:
    """Build a security-hardened ML CI workflow with all best practices applied."""

    train_step = {
        "name": "Train model",
        "run": "python train.py --output model.pkl",
    }

    steps = [
        # Pinned to SHA for supply chain integrity
        {"uses": "actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683"},  # v4.2.2
        {
            "uses": "actions/setup-python@0b93645e9fea7318ecaed2b359559ac225c90a2b",  # v5.3.0
            "with": {"python-version": python_version},
        },
        {
            "name": "Install dependencies",
            "run": "pip install -r requirements.txt",
        },
        {"name": "Run tests", "run": "pytest tests/"},
        train_step,
    ]

    permissions = {"contents": "read"}

    if use_oidc:
        permissions["id-token"] = "write"
        steps.append(
            {
                "name": "Configure AWS (OIDC — no stored key)",
                "uses": "aws-actions/configure-aws-credentials@v4",
                "with": {
                    "role-to-assume": aws_role_arn,
                    "aws-region": aws_region,
                },
            }
        )
        steps.append(
            {
                "name": "Upload model to S3",
                # Secret only needed for model registry auth — passed via env
                "env": {"MODEL_REGISTRY_TOKEN": "${{ secrets.MODEL_REGISTRY_TOKEN }}"},
                "run": "aws s3 cp model.pkl s3://models/latest.pkl",
            }
        )

    return {
        "name": "Secure ML CI",
        "on": {"push": {"branches": ["main"]}},
        "jobs": {
            "train-and-deploy": {
                "runs-on": "ubuntu-latest",
                "permissions": permissions,
                "steps": steps,
            }
        },
    }


print(yaml.dump(build_secure_ml_workflow(), sort_keys=False))


**What just happened?**

- We composed a complete secure workflow programmatically — useful for generating workflows in a monorepo.
- Every security practice is applied: SHA pins, scoped permissions, env-var secrets, OIDC.
- The `MODEL_REGISTRY_TOKEN` secret is only in `env:` — never interpolated into the `run:` string.


## Step 7 · Environment Secrets and Deployment Protection Rules

For production deployments, GitHub **Environments** add an extra layer: required reviewers, wait timers, and environment-scoped secrets that are only available to jobs that reference that environment.

| Feature | Effect |
|---|---|
| Required reviewers | Job waits for manual approval before running |
| Wait timer | Minimum minutes before deployment proceeds |
| Branch protection | Only specific branches can deploy to this environment |
| Environment secrets | Secrets scoped to this environment only |


In [ ]:
# Production deployment using a protected environment
env_workflow = {
    "name": "Protected production deploy",
    "on": {"push": {"branches": ["main"]}},
    "jobs": {
        "deploy-staging": {
            "runs-on": "ubuntu-latest",
            # Staging: no protection rules, environment-scoped secrets
            "environment": "staging",
            "permissions": {"contents": "read", "id-token": "write"},
            "steps": [
                {"uses": "actions/checkout@v4"},
                {
                    "name": "Deploy to staging",
                    # STAGING_API_KEY only available in the 'staging' environment
                    "env": {"API_KEY": "${{ secrets.STAGING_API_KEY }}"},
                    "run": "python deploy.py --env staging",
                },
            ],
        },
        "deploy-production": {
            "runs-on": "ubuntu-latest",
            "needs": "deploy-staging",
            # Production: protected — requires manual approval
            "environment": "production",
            "permissions": {"contents": "read", "id-token": "write"},
            "steps": [
                {"uses": "actions/checkout@v4"},
                {
                    "name": "Deploy to production",
                    # PROD_API_KEY only available in the 'production' environment
                    "env": {"API_KEY": "${{ secrets.PROD_API_KEY }}"},
                    "run": "python deploy.py --env production",
                },
            ],
        },
    },
}

print(yaml.dump(env_workflow, sort_keys=False))


**What just happened?**

- The `production` environment is configured in GitHub Settings to require manual approval — the job pauses until a reviewer clicks "Approve".
- `PROD_API_KEY` is a secret scoped to the `production` environment only — it cannot be read by any job that doesn't specify `environment: production`.
- This pattern is called **progressive delivery**: stage → approve → production.


## Step 8 · Security Audit: Scanning a Workflow for Common Issues

Let's write a Python function that parses a workflow YAML and flags common security issues. This mirrors what tools like `actionlint` and `zizmor` do.


In [ ]:
import re


def audit_workflow(workflow_yaml: str) -> list[str]:
    """Scan a workflow YAML string for common security issues."""
    issues = []
    wf = yaml.safe_load(workflow_yaml)

    for job_name, job in (wf.get("jobs") or {}).items():
        # Check: no permissions key = inherits repo defaults (potentially wide)
        if "permissions" not in job:
            issues.append(f"[{job_name}] No 'permissions:' key — inherits repo-level defaults")

        for i, step in enumerate(job.get("steps") or []):
            uses = step.get("uses", "")
            run = step.get("run", "")

            # Check: action pinned to tag, not SHA
            if uses and "@" in uses:
                ref = uses.split("@")[1]
                if not re.match(r'^[0-9a-f]{40}$', ref):
                    issues.append(
                        f"[{job_name}] step {i+1} uses '{uses}' — pin to a commit SHA for supply chain safety"
                    )

            # Check: secret interpolated directly in run string
            if run and "secrets." in run:
                issues.append(
                    f"[{job_name}] step {i+1} interpolates a secret in 'run:' — use 'env:' instead"
                )

    return issues


# Test on a deliberately insecure workflow
insecure_wf = """
name: Insecure example
on: push
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Deploy
        run: deploy.sh --token ${{ secrets.API_TOKEN }}
"""

findings = audit_workflow(insecure_wf)
print("Security audit findings:")
for f in findings:
    print(f"  ⚠  {f}")
if not findings:
    print("  ✓ No issues found")


**What just happened?**

- Our auditor found two issues: no `permissions:` key and a secret interpolated in `run:`.
- This is a simplified version of what `actionlint` and `zizmor` do — real tools also check expression injection, workflow_dispatch inputs, and more.
- **Run `actionlint` in CI** to catch these before they reach production workflows.


In [ ]:
# Challenge: Harden this workflow
#
# The workflow below has multiple security issues:
#   1. No permissions key — inherits repo defaults
#   2. Secret used directly in run: string
#   3. Actions pinned to mutable tags
#   4. Top-level permissions not restricted
#
# Fix all issues. Then verify with the audit_workflow() function above.
#
# Bonus: add an OIDC step to replace the stored DEPLOY_KEY secret entirely.

insecure_to_fix = """\
name: Deploy model
on:
  push:
    branches: [main]
jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - name: Train model
        run: python train.py
      - name: Upload model
        run: curl -H 'Authorization: ${{ secrets.DEPLOY_KEY }}' -F file=@model.pkl https://registry.example.com/upload
"""

# TODO: create fixed_workflow string with all issues resolved
fixed_workflow = insecure_to_fix  # replace this

# Verify your fix:
findings = audit_workflow(fixed_workflow)
if findings:
    print("Still has issues:")
    for f in findings:
        print(f"  ⚠  {f}")
else:
    print("✓ All security issues resolved!")


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| Secrets via `env:` | Never interpolate `${{ secrets.X }}` in `run:` — map to env vars |
| `GITHUB_TOKEN` scoping | Set `permissions:` at job level to least privilege |
| SHA pinning | Third-party actions should be pinned to full 40-char commit SHA |
| OIDC | Eliminates long-lived credentials — credentials expire with the job |
| Environments | Add approval gates and scope secrets to deployment targets |
| `actionlint` / `zizmor` | Lint workflows in CI to catch secret leaks and injection risks |

> **Tip:** Never echo a secret directly. Pass secrets via environment variables to subprocess calls, never via positional arguments.

---
## What's next
**Day 6** → ML-specific CI/CD patterns — training sklearn models in Actions, uploading artifacts to GitHub Releases, evaluation gates, and posting PR comments with results.

Mark Day 5 complete in your [tracker](../index.html).
